# 67. 누락가스 군집 전문가 앙상블 — 재현율 우선 중첩 구간검증
66번의전체개발군집을평가에쓰지않고각외부학습구간에서가스모드3개를새로학습한다. 정상대체·스케일·군집·트리는참조학습만사용한다. 내부구간제외정상상위2/5/10%경계로판정하고학습내부재현율우선,F1다음으로선택한다. 이미본620개(가스19)를이용한알고리즘검증으로완전히새로운표본독립검증아니다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
# Reuse only sensor preparation from63, never sealed predictions or its fitting.
s=''.join(json.loads((root/'notebooks/63_rg3_blind_frozen_extra37_predictions.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('# Original 47 innovation expert')[0])
known=pd.read_csv(tables/'rg3_independent37_unblinded_results.csv',parse_dates=['TimeStamp']).set_index('TimeStamp')
d.loc[n_train:,'PassOrFail']=known.loc[d.TimeStamp.iloc[n_train:],'actual_gas'].to_numpy(int)
y=d.PassOrFail.to_numpy(int)
d=d.sort_values('TimeStamp').reset_index(drop=True);y=d.PassOrFail.to_numpy(int)
gap=d.TimeStamp.diff().dt.total_seconds()
groups=(gap.gt(1800)|gap.isna()).cumsum()
blocks=groups.unique()
F=pd.DataFrame({
 'pressure_switch_delta':d.Max_Injection_Pressure__delta1-d.Max_Switch_Over_Pressure__delta1,
 'barrel_mold_delta':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'cushion_delta':d.Cushion_Position__delta1,
 'hopper_delta':d.Hopper_Temperature__delta1,
 'plasticizing_time_delta':d.Plasticizing_Time__delta1,
 'meter_position_delta':d.Plasticizing_Position__delta1}).replace([np.inf,-np.inf],np.nan)
imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(F[y==0])
sc=RobustScaler().fit(imp.transform(F[y==0]))
z=sc.transform(imp.transform(F))
diagnostic=KMeans(n_clusters=3,n_init=20,random_state=42).fit_predict(z[y==1])
gas=d.loc[y==1,['TimeStamp']].copy();gas['diagnostic_cluster']=diagnostic
gas=pd.concat([gas.reset_index(drop=True),F[y==1].reset_index(drop=True)],axis=1)
gas.to_csv(tables/'rg3_gas_positive_mode_diagnostic.csv',index=False,encoding='utf-8-sig')
print('Descriptive gas-only modes; NOT used globally in evaluated models')
print(gas.to_string(index=False))
memo={}
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),2))
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(F.iloc[ref][y[ref]==0])
 tr=im.transform(F.iloc[ref]);te=im.transform(F.iloc[q])
 scale=RobustScaler().fit(tr[y[ref]==0]);tr=scale.transform(tr);te=scale.transform(te)
 pos=np.flatnonzero(y[ref]==1);normal=np.flatnonzero(y[ref]==0)
 k=min(3,len(np.unique(tr[pos],axis=0)))
 labels=KMeans(n_clusters=k,n_init=20,random_state=42).fit_predict(tr[pos]) if k>1 else np.zeros(len(pos),int)
 modes=[];global_models=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7]:
   model=cls(n_estimators=100,max_depth=5,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2)
   global_models.append(model.fit(tr,y[ref]).predict_proba(te)[:,1])
   sub=[]
   for mode in np.unique(labels):
    idx=np.r_[normal,pos[labels==mode]]
    model=cls(n_estimators=100,max_depth=5,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2)
    sub.append(model.fit(tr[idx],y[ref][idx]).predict_proba(te)[:,1])
   modes.append(np.max(sub,axis=0))
 memo[key]=np.column_stack([np.mean(global_models,axis=0),np.mean(modes,axis=0)])
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['global_ensemble','gas_mode_max_ensemble']):
  for budget in [.02,.05,.1]:
   norm=inner[y[ref]==0,k]
   cut=float(np.quantile(norm,1-budget,method='higher'))
   pp=inner[:,k]>cut
   tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0
   f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished recall-first outer block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
p.to_csv(tables/'rg3_recall_first_mode_predictions.csv',index=False,encoding='utf-8-sig')
m.to_csv(tables/'rg3_recall_first_mode_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_recall_first_mode_inner_choices.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Known-data nested algorithm evaluation, NOT untouched independent validation.')
